# Defining and Using LangChain Tools

This reference notebook shows how to turn a typed Python function into a LangChain tool and then give that tool to an agent.

## Learning goals

- Define a tool with the `@tool` decorator.
- Compare the decorator's default, named, and explicitly described forms.
- Invoke a tool directly with structured input.
- Let an agent decide when to call a tool and inspect the resulting message history.

## Before you run the notebook

1. Configure the model-provider API key in the environment or `.env` file.
2. Run the cells from top to bottom.
3. The examples calculate square roots only. A negative input is rejected because the tool is designed to return real-valued results.

The final tool definition is the canonical version used by the agent. The earlier definitions are included to make the decorator options easy to compare.

# Tool Definition

In [ ]:
# Load API keys and other local settings from the project's .env file.
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import math

from langchain.tools import tool


@tool
def square_root(x: float) -> float:
    """Calculate the real square root of a non-negative number."""
    if x < 0:
        raise ValueError("square_root requires a non-negative number")
    return math.sqrt(x)

In [3]:
# A tool name can be supplied explicitly when the Python function name is not ideal.
@tool("square_root")
def named_square_root(x: float) -> float:
    """Calculate the real square root of a non-negative number."""
    if x < 0:
        raise ValueError("square_root requires a non-negative number")
    return math.sqrt(x)

In [4]:
# A custom description helps the model understand when this tool is appropriate.
@tool("square_root", description="Calculate the real square root of a non-negative number")
def described_square_root(x: float) -> float:
    if x < 0:
        raise ValueError("square_root requires a non-negative number")
    return math.sqrt(x)

# Use the explicitly described version in the remaining examples.
tool1 = described_square_root

In [5]:
# Direct tool calls use a dictionary keyed by the function argument name.
tool1.invoke({"x": 467})

21.61018278497431

### Tool-definition summary

The decorator converts a Python function into a tool object with a name, description, and input schema. The explicit description is important because the model uses it to decide whether the tool matches a request. Direct invocation is a useful first test because it isolates the tool from agent behavior.

## 2. Add the Tool to an Agent

Once the tool works independently, pass it in the agent's `tools` list. The agent can then choose the tool when the user's request requires a square-root calculation.

In [ ]:
from langchain.agents import create_agent

# The tool description tells the model what this tool can calculate.
agent = create_agent(
    model="gpt-5-nano",
    tools=[tool1],
    system_prompt="You are an arithmetic assistant. Use the square_root tool for square-root calculations.",
)

In [ ]:
from langchain.messages import HumanMessage

question = HumanMessage(content="What is the square root of 467?")

# The agent may call square_root before returning its final answer.
response = agent.invoke({"messages": [question]})

print(response["messages"][-1].content)

The square root of 467 is approximately 21.61018278497431.


In [ ]:
from pprint import pprint

# Inspect the full message history, including any tool call and tool result.
pprint(response["messages"])

[HumanMessage(content='What is the square root of 467?', additional_kwargs={}, response_metadata={}, id='696b35f5-5dda-4bcb-ba8f-1ba493048b2e'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1818, 'prompt_tokens': 158, 'total_tokens': 1976, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 1792, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EPYbPVCROlkreCfLOJ2249zyziXen', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0b601-0e21-7b72-9d33-188c8acdadc7-0', tool_calls=[{'name': 'square_root', 'args': {'x': 467}, 'id': 'call_qmfXRAm8WVfzQsMtnDjbKGUf', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 158, '

In [ ]:
# The final message is the assistant's user-facing response.
print(response["messages"][-1].content)

The square root of 467 is approximately 21.61018278497431.


In [ ]:
# Tool calls belong to an earlier assistant message, so inspect the full history.
tool_calls = [
    call
    for message in response["messages"]
    for call in getattr(message, "tool_calls", [])
]
print(tool_calls)

[]


### Agent section summary

The agent receives the tool schema and can request a calculation through a structured tool call. Inspecting all returned messages lets you verify the intermediate tool call, its arguments, its result, and the final response instead of assuming the last message contains every detail.

## Conclusion and reference checklist

A LangChain tool is a typed Python function exposed to the model with a name, description, and input schema. A reliable development workflow is:

1. Define the smallest useful function and validate its inputs.
2. Describe the tool clearly so the model can select it appropriately.
3. Invoke the tool directly to test its own behavior.
4. Add it to an agent and test the complete tool-call loop.
5. Inspect the full message history when debugging tool selection or results.

For production tools, add domain-specific error handling, input limits, logging, and tests for invalid inputs. Keep the tool's documented behavior aligned with what the function actually returns.